In [ ]:
# Cell 1: Dependencies
#!pip install google-generativeai datasets pandas pyarrow huggingface_hub langid

# Cell 2: Imports
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset, DatasetDict
import os
import json
import time
import google.generativeai as genai
from datetime import datetime
import re
import langid
from collections import Counter
import hashlib
from huggingface_hub import login, upload_file, hf_hub_download

In [ ]:
# Cell 3: Corpus Domain Configuration (EASY TO MODIFY)
DOMAIN_CONFIG = {
    'domain': 'Indonesian Legal Corpus Analysis',
    'target_language': 'id',
    'num_variants': 2,  # QA variants per corpus chunk
    'approach': 'iterative_thinking',  # 'simple', 'deep_thinking', 'iterative_thinking'

    # Single shot processing settings
    'max_text_length': 15000,  # Maximum characters to send to Gemini (adjust as needed)
    'min_text_length': 200,    # Minimum text length to process
    
    # Corpus processing settings
    'chunk_method': 'sentences',  # 'sentences', 'words', 'paragraphs'
    'chunk_size': 30,  # sentences per chunk (adjust based on method)
    'overlap': 1,  # overlap between chunks
    'min_chunk_length': 100,  # minimum characters per chunk
}

# System prompts for different approaches
SYSTEM_PROMPTS = {
    'simple': """Anda adalah asisten yang membuat pertanyaan-jawaban berdasarkan teks.""",
    'deep_thinking': """Anda adalah asisten yang membuat pertanyaan dengan proses analisis step-by-step.""",
    'iterative_thinking': """Anda adalah asisten yang membuat pertanyaan dengan proses berpikir bertahap yang detail."""
}

# Separate simple prompts to avoid safety filters
SIMPLE_PROMPTS = {
    'question_generation': '''Buat {num_variants} pertanyaan berbeda tentang teks ini:

{text_preview}

Format JSON sederhana:
[
  {{"question": "pertanyaan 1"}},
  {{"question": "pertanyaan 2"}}
]''',

    'answer_generation': '''Jawab pertanyaan ini berdasarkan teks:

TEKS: {text_preview}
PERTANYAAN: {question}

Jawaban (200-400 kata):''',

    'thinking_generation': '''Buat proses analisis step-by-step untuk menjawab pertanyaan ini:

TEKS: {text_preview}
PERTANYAAN: {question}

Proses analisis (minimal 300 kata):
1. Analisis awal:
2. Identifikasi poin utama:
3. Pertimbangan berbagai aspek:
4. Evaluasi evidence:
5. Kesimpulan:'''
}

# Combined prompts (as backup)
COMBINED_PROMPTS = {
    'simple': '''Buat {num_variants} pertanyaan-jawaban sederhana tentang teks ini:

{full_text}

Format:
[
  {{"question": "pertanyaan 1", "answer": "jawaban 1"}},
  {{"question": "pertanyaan 2", "answer": "jawaban 2"}}
]''',

    'deep_thinking': '''Buat {num_variants} pertanyaan dengan analisis singkat:

{full_text}

Format:
[
  {{"question": "pertanyaan 1", "thinking": "analisis step-by-step minimal 200 kata", "answer": "jawaban 1"}},
  {{"question": "pertanyaan 2", "thinking": "proses berpikir minimal 200 kata", "answer": "jawaban 2"}}
]''',

    'iterative_thinking': '''Buat {num_variants} pertanyaan dengan proses berpikir detail:

{full_text}

Format:
[
  {{"question": "pertanyaan 1", "thinking": "Step 1: Analisis... Step 2: Evaluasi... Step 3: Sintesis... (minimal 400 kata)", "answer": "jawaban singkat 1"}},
  {{"question": "pertanyaan 2", "thinking": "Langkah 1: Pemahaman... Langkah 2: Assessment... Langkah 3: Konklusi... (minimal 400 kata)", "answer": "jawaban langsung 2"}}
]'''
}

In [ ]:
# Cell 4: Main Configuration
CONFIG = {
    #'gemini_api_key': '',
    'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    'huggingface_token': '',
    'source_dataset': 'joelniklaus/legal_case_document_summarization',  # Change this
    'output_repository': 'Azzindani/Legal_Corpus_QA_SynDeepThink',  # Change this
    'corpus_column': 'judgement',  # Column name containing the corpus text
    
    # Processing settings
    'batch_size': 50,  # Corpus chunks per processing batch
    'model_name': 'gemini-2.5-flash',
    'temperature': 0.8,
    'base_tokens_per_variant': 5000,  # Adjust based on approach
    
    # Quality settings
    'min_question_length': 20,
    'min_answer_length': 50,
    'max_question_length': 3000,
    'max_answer_length': 6000,
    'target_language_confidence': 0.5,
    
    # Corpus specific settings
    'skip_processed_rows': True,
    'progress_file': 'synthesis_progress.json'
}

In [ ]:
# Cell 5: Authentication
genai.configure(api_key=CONFIG['gemini_api_key'])
login(token=CONFIG['huggingface_token'])

In [ ]:
# Cell 6: Text Validator for Single-Shot Processing
class TextValidator:
    def __init__(self):
        self.max_length = DOMAIN_CONFIG['max_text_length']
        self.min_length = DOMAIN_CONFIG['min_text_length']
    
    def validate_and_prepare_text(self, text):
        """Validate and prepare text for single-shot processing"""
        if not text or not isinstance(text, str):
            return None
        
        text = text.strip()
        
        if len(text) < self.min_length:
            return None
        
        # Truncate if too long (preserve beginning and end)
        if len(text) > self.max_length:
            half_max = self.max_length // 2 - 100  # Leave room for separator
            text = text[:half_max] + "\n\n[...TRUNCATED...]\n\n" + text[-half_max:]
        
        return text

In [ ]:
# Cell 7: Language Detection (same as before)
class LanguageScorer:
    def __init__(self, target_language='id'):
        self.target_language = target_language
        langid.set_languages([target_language, 'en'])
    
    def detect_language(self, text):
        try:
            lang, confidence = langid.classify(text)
            return lang, confidence
        except:
            return 'unknown', 0.0
    
    def score_language_accuracy(self, question, answer):
        q_lang, q_conf = self.detect_language(question)
        a_lang, a_conf = self.detect_language(answer)
        
        scores = {
            'question_language': q_lang,
            'question_confidence': q_conf,
            'answer_language': a_lang,
            'answer_confidence': a_conf,
            'question_correct_language': q_lang == self.target_language,
            'answer_correct_language': a_lang == self.target_language,
            'language_accuracy': (q_conf + a_conf) / 2,
            'passes_language_check': True  # Always pass for corpus data
        }
        
        return scores

In [ ]:
# Cell 8: Progress Manager (adapted for corpus)
class ProgressManager:
    def __init__(self):
        self.progress_data = {
            'processed_chunks': [],
            'current_row': 0,
            'total_chunks_processed': 0,
            'total_qa_pairs_created': 0,
            'request_count': 0,
            'start_time': None,
            'last_update': None,
            'errors': [],
            'statistics': {
                'avg_chunks_per_text': 0.0,
                'avg_qa_per_chunk': 0.0,
                'approach_stats': {'simple': 0, 'deep_thinking': 0, 'iterative_thinking': 0}
            }
        }
        self.load_progress()
    
    def load_progress(self):
        try:
            progress_path = hf_hub_download(
                repo_id=CONFIG['output_repository'],
                filename=CONFIG['progress_file'],
                repo_type="dataset"
            )
            with open(progress_path, 'r') as f:
                saved_progress = json.load(f)
                self.progress_data.update(saved_progress)
            print(f"Progress loaded: {self.progress_data['total_chunks_processed']} chunks processed")
        except Exception as e:
            print(f"No existing progress found, starting fresh: {e}")
            self.progress_data['start_time'] = datetime.now().isoformat()
    
    def save_progress(self):
        try:
            self.progress_data['last_update'] = datetime.now().isoformat()
            
            def convert_types(obj):
                if isinstance(obj, dict):
                    return {k: convert_types(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_types(v) for v in obj]
                elif hasattr(obj, 'item'):
                    return obj.item()
                elif hasattr(obj, 'tolist'):
                    return obj.tolist()
                else:
                    return obj
            
            clean_data = convert_types(self.progress_data)
            local_path = f"./{CONFIG['progress_file']}"
            
            with open(local_path, 'w') as f:
                json.dump(clean_data, f, indent=2)
            
            upload_file(
                path_or_fileobj=local_path,
                path_in_repo=CONFIG['progress_file'],
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Corpus progress: {self.progress_data['total_qa_pairs_created']} QA pairs"
            )
            print(f"Progress saved to repository")
        except Exception as e:
            print(f"Failed to save progress: {e}")
    
    def is_chunk_processed(self, chunk_id):
        return chunk_id in self.progress_data['processed_chunks']
    
    def mark_chunk_processed(self, chunk_id, qa_count):
        if chunk_id not in self.progress_data['processed_chunks']:
            self.progress_data['processed_chunks'].append(chunk_id)
            self.progress_data['total_chunks_processed'] += 1
            self.progress_data['total_qa_pairs_created'] += qa_count

In [ ]:
# Cell 9: Multi-Step Corpus Synthesizer
class MultiStepCorpusSynthesizer:
    def __init__(self, progress_manager):
        # Much higher token limit for complete thinking processes
        
        self.model = genai.GenerativeModel(
            CONFIG['model_name'],
            generation_config=genai.types.GenerationConfig(
                temperature=CONFIG['temperature'],
                max_output_tokens=CONFIG['max_answer_length'],  # Higher limit
                top_p=0.9,
                top_k=40
            ),
            system_instruction=SYSTEM_PROMPTS[DOMAIN_CONFIG['approach']]
        )
        
        self.language_scorer = LanguageScorer(DOMAIN_CONFIG['target_language'])
        self.progress_manager = progress_manager
        self.text_validator = TextValidator()
        self.request_count = progress_manager.progress_data['request_count']
    
    def generate_questions_only(self, text_preview):
        """Step 1: Generate questions only"""
        prompt = SIMPLE_PROMPTS['question_generation'].format(
            num_variants=DOMAIN_CONFIG['num_variants'],
            text_preview=text_preview[:3000]
        )
        
        try:
            time.sleep(3)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates or response.candidates[0].finish_reason == 2:
                return []
            
            response_text = response.candidates[0].content.parts[0].text.strip()
            
            # Extract questions
            try:
                json_match = re.search(r'\[.*?\]', response_text, re.DOTALL)
                if json_match:
                    questions_data = json.loads(json_match.group(0))
                    return [q.get('question', '').strip() for q in questions_data if q.get('question')]
            except:
                # Fallback: extract from text
                questions = re.findall(r'"question":\s*"([^"]+)"', response_text)
                return [q for q in questions if len(q) > 10]
            
            return []
            
        except Exception as e:
            print(f"    Question generation error: {e}")
            return []
    
    def generate_answer_for_question(self, text_preview, question):
        """Step 2: Generate answer for specific question"""
        prompt = SIMPLE_PROMPTS['answer_generation'].format(
            text_preview=text_preview[:4000],
            question=question
        )
        
        try:
            time.sleep(3)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates or response.candidates[0].finish_reason == 2:
                return None
            
            answer = response.candidates[0].content.parts[0].text.strip()
            return answer if len(answer) > 30 else None
            
        except Exception as e:
            print(f"    Answer generation error: {e}")
            return None
    
    def generate_thinking_for_question(self, text_preview, question):
        """Step 3: Generate thinking process - handle long responses"""
        if DOMAIN_CONFIG['approach'] == 'simple':
            return ""
        
        # More specific prompt with length guidance
        prompt = f'''Buat analisis bertahap yang lengkap untuk pertanyaan ini:
    
    TEKS: {text_preview[:4000]}
    PERTANYAAN: {question}
    
    Buat analisis dalam format berikut (masing-masing bagian minimal 100 kata):
    
    LANGKAH 1 - ANALISIS AWAL:
    [Analisis pemahaman awal terhadap teks dan pertanyaan]
    
    LANGKAH 2 - IDENTIFIKASI POIN UTAMA:
    [Identifikasi elemen-elemen kunci dari teks yang relevan]
    
    LANGKAH 3 - PERTIMBANGAN MULTI-ASPEK:
    [Evaluasi dari berbagai sudut pandang dan perspektif]
    
    LANGKAH 4 - EVALUASI KRITIS:
    [Cross-checking, validasi, dan assessment evidence]
    
    LANGKAH 5 - SINTESIS KESIMPULAN:
    [Konsolidasi semua analisis menjadi pemahaman terintegrasi]
    
    Berikan analisis lengkap tanpa pemotongan.'''
        
        try:
            time.sleep(4)  # Longer wait for complex generation
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates:
                return self.try_combined_approach(text_preview, question)
            
            candidate = response.candidates[0]
            
            if candidate.finish_reason == 2:  # Safety
                print(f"    Thinking generation filtered, trying alternative")
                return self.try_alternative_thinking(text_preview, question)
            
            if candidate.finish_reason == 3:  # Length limit reached
                print(f"    Thinking truncated by length limit, trying to get continuation")
                partial_thinking = candidate.content.parts[0].text.strip()
                continuation = self.get_thinking_continuation(text_preview, question, partial_thinking)
                full_thinking = partial_thinking + "\n\n" + continuation if continuation else partial_thinking
                print(f"    Combined thinking: {len(full_thinking)} chars")
                return full_thinking
            
            if not candidate.content or not candidate.content.parts:
                return ""
            
            thinking = candidate.content.parts[0].text.strip()
            print(f"    Generated complete thinking: {len(thinking)} chars")
            
            # Check if thinking appears to be cut off
            if len(thinking) > 1000 and (thinking.endswith('---') or thinking.endswith('...') or re.search(r'\d+\.\s*$', thinking)):
                print(f"    Thinking appears truncated, attempting continuation...")
                continuation = self.get_thinking_continuation(text_preview, question, thinking)
                if continuation:
                    thinking = thinking + "\n\n" + continuation
                    print(f"    Extended thinking: {len(thinking)} chars")
            
            return thinking if len(thinking) > 200 else ""
            
        except Exception as e:
            print(f"    Thinking generation error: {e}")
            return self.try_combined_approach(text_preview, question)
    
    def get_thinking_continuation(self, text_preview, question, partial_thinking):
        """Get continuation of truncated thinking"""
        continuation_prompt = f'''Lanjutkan analisis yang terpotong ini:
    
    TEKS: {text_preview[:3000]}
    PERTANYAAN: {question}
    
    ANALISIS YANG SUDAH ADA:
    {partial_thinking[-500:]}
    
    Lanjutkan analisis dari titik terakhir hingga selesai dengan:
    - Melengkapi langkah yang terpotong
    - Menyelesaikan semua langkah analisis yang tersisa
    - Memberikan kesimpulan final yang lengkap
    
    Lanjutan analisis:'''
        
        try:
            time.sleep(3)
            response = self.model.generate_content(continuation_prompt)
            if response.candidates and response.candidates[0].finish_reason not in [2, 4]:
                continuation = response.candidates[0].content.parts[0].text.strip()
                print(f"    Got continuation: {len(continuation)} chars")
                return continuation
        except:
            pass
        
        return ""
    
    def try_alternative_thinking(self, text_preview, question):
        """Alternative approach when main thinking is filtered"""
        simple_prompt = f'''Analisis sederhana untuk pertanyaan ini:
    
    PERTANYAAN: {question}
    TEKS: {text_preview[:3000]}
    
    Berikan analisis dalam poin-poin:
    1. Pemahaman dasar:
    2. Poin-poin utama:
    3. Pertimbangan penting:
    4. Evaluasi:
    5. Kesimpulan:
    
    Analisis lengkap:'''
        
        try:
            time.sleep(3)
            response = self.model.generate_content(simple_prompt)
            if response.candidates and response.candidates[0].finish_reason not in [2, 4]:
                thinking = response.candidates[0].content.parts[0].text.strip()
                print(f"    Alternative thinking: {len(thinking)} chars")
                return thinking
            return ""
        except:
            return ""
    
    def process_corpus_row(self, text_row, row_index):
        """Process corpus row using multi-step approach"""
        text_content = text_row[CONFIG['corpus_column']]
        
        prepared_text = self.text_validator.validate_and_prepare_text(text_content)
        if not prepared_text:
            print(f"Row {row_index}: Invalid text, skipping")
            return []
        
        print(f"Processing row {row_index}: {len(prepared_text)} chars using MULTI-STEP approach")
        
        # Step 1: Generate questions
        questions = self.generate_questions_only(prepared_text)
        if not questions:
            print(f"  No questions generated for row {row_index}")
            return []
        
        print(f"  Step 1: Generated {len(questions)} questions")
        
        results = []
        for q_idx, question in enumerate(questions):
            if len(question.strip()) < 15:
                continue
            
            print(f"    Processing Q{q_idx+1}: {question[:50]}...")
            
            # Step 2: Generate answer
            answer = self.generate_answer_for_question(prepared_text, question)
            if not answer or len(answer) < 30:
                print(f"      No valid answer generated")
                continue
            
            # Step 3: Generate thinking (if needed)
            thinking = ""
            if DOMAIN_CONFIG['approach'] in ['deep_thinking', 'iterative_thinking']:
                thinking = self.generate_thinking_for_question(prepared_text, question)
            
            # Score language
            content_for_scoring = f"{question} {thinking} {answer}" if thinking else f"{question} {answer}"
            language_scores = self.language_scorer.score_language_accuracy(question, content_for_scoring)
            
            result = {
                'source_row_index': row_index,
                'variant_index': q_idx,
                'source_text_length': len(prepared_text),
                'source_text_preview': prepared_text[:300],
                'question': question,
                'answer': answer,
                'approach': DOMAIN_CONFIG['approach'],
                'processing_method': 'multi_step',
                'question_length': len(question),
                'answer_length': len(answer),
                'question_word_count': len(question.split()),
                'answer_word_count': len(answer.split()),
                'timestamp': datetime.now().isoformat(),
                **language_scores
            }
            
            if thinking:
                result['thinking'] = thinking
                result['thinking_length'] = len(thinking)
                result['thinking_word_count'] = len(thinking.split())
                print(f"      ✓ Complete: Q+T({len(thinking)})+A")
            else:
                result['thinking'] = ''
                result['thinking_length'] = 0
                result['thinking_word_count'] = 0
                print(f"      ✓ Complete: Q+A (no thinking)")
            
            results.append(result)
        
        print(f"  ✓ Row {row_index}: {len(results)} complete QA pairs")
        return results

In [ ]:
# Cell 10: Main Processing Functions with HF Dataset Compatibility
def load_corpus_dataset():
    """Load the corpus dataset"""
    try:
        dataset = load_dataset(CONFIG['source_dataset'], trust_remote_code=True)
        if isinstance(dataset, dict):
            ds = dataset['train'] if 'train' in dataset else dataset[list(dataset.keys())[0]]
        else:
            ds = dataset
        
        print(f"Corpus dataset loaded: {len(ds)} rows")
        print(f"Columns: {list(ds[0].keys())}")
        print(f"Using column: '{CONFIG['corpus_column']}'")
        
        if CONFIG['corpus_column'] not in ds[0]:
            print(f"ERROR: Column '{CONFIG['corpus_column']}' not found in dataset")
            print(f"Available columns: {list(ds[0].keys())}")
            return None
        
        return ds
    except Exception as e:
        print(f"Error loading corpus dataset: {e}")
        return None

def process_corpus_batch(start_row, end_row=None):
    """Process a batch of corpus rows with HF-compatible output"""
    corpus_ds = load_corpus_dataset()
    if not corpus_ds:
        return
    
    if end_row is None:
        end_row = min(start_row + CONFIG['batch_size'], len(corpus_ds))
    
    # With:
    synthesizer = MultiStepCorpusSynthesizer(progress_manager)
    
    # And change the log message:
    print(f"MULTI-STEP Processing corpus rows {start_row} to {end_row-1}")
    
    all_results = []
    for row_idx in range(start_row, end_row):
        if row_idx >= len(corpus_ds):
            break
        
        if CONFIG['skip_processed_rows'] and str(row_idx) in progress_manager.progress_data.get('processed_rows', []):
            print(f"Row {row_idx} already processed, skipping")
            continue
        
        try:
            text_row = corpus_ds[row_idx]
            results = synthesizer.process_corpus_row(text_row, row_idx)
            
            if results:
                all_results.extend(results)
                if 'processed_rows' not in progress_manager.progress_data:
                    progress_manager.progress_data['processed_rows'] = []
                progress_manager.progress_data['processed_rows'].append(str(row_idx))
                progress_manager.progress_data['total_qa_pairs_created'] += len(results)
            
            progress_manager.save_progress()
            
        except Exception as e:
            print(f"Error processing row {row_idx}: {e}")
            continue
    
    if all_results:
        save_standardized_batch(all_results, start_row, end_row)
    
    print(f"Batch completed: {len(all_results)} QA pairs from {end_row - start_row} corpus rows")

def save_standardized_batch(results, start_row, end_row):
    """Save batch with HF Dataset compatible format and naming"""
    if not results:
        return
    
    try:
        # Standardize data structure for HF compatibility
        standardized_data = []
        
        for result in results:
            # Create consistent schema regardless of approach
            standard_record = {
                # Core fields
                'question': result['question'],
                'answer': result['answer'],
                
                # Metadata fields - consistent naming
                'source_row': result['source_row_index'],
                'variant_id': result['variant_index'],
                'approach': result['approach'],
                
                # Length metrics
                'question_length': result['question_length'],
                'answer_length': result['answer_length'],
                'question_words': result['question_word_count'],
                'answer_words': result['answer_word_count'],
                
                # Language detection
                'question_lang': result['question_language'],
                'answer_lang': result['answer_language'],
                'lang_confidence_q': float(result['question_confidence']),
                'lang_confidence_a': float(result['answer_confidence']),
                
                # Source info
                'source_preview': result['source_text_preview'][:200],  # Limit length
                'source_length': result['source_text_length'],
                'processing_type': 'single_shot',
                'timestamp': result['timestamp']
            }
            
            # Add thinking field only if it exists
            if 'thinking' in result and result['thinking']:
                standard_record['thinking'] = result['thinking']
                standard_record['thinking_length'] = result.get('thinking_length', 0)
                standard_record['thinking_words'] = result.get('thinking_word_count', 0)
            else:
                # Always include these fields for consistency
                standard_record['thinking'] = ''
                standard_record['thinking_length'] = 0
                standard_record['thinking_words'] = 0
            
            standardized_data.append(standard_record)
        
        # Create DataFrame with consistent schema
        df = pd.DataFrame(standardized_data)
        
        # Use HF standard naming format - short and compatible
        batch_id = f"{start_row:05d}_{end_row:05d}"
        filename = f"train-{batch_id}.parquet"  # HF standard format
        
        temp_filepath = f"/tmp/{filename}"
        
        # Save with consistent dtypes
        df = df.astype({
            'question': 'string',
            'answer': 'string',
            'thinking': 'string',
            'approach': 'string',
            'question_lang': 'string',
            'answer_lang': 'string',
            'source_preview': 'string',
            'processing_type': 'string',
            'timestamp': 'string'
        })
        
        df.to_parquet(temp_filepath, index=False, engine='pyarrow')
        
        # Upload to repository
        upload_file(
            path_or_fileobj=temp_filepath,
            path_in_repo=filename,
            repo_id=CONFIG['output_repository'],
            repo_type="dataset",
            commit_message=f"Batch {batch_id}: {len(standardized_data)} QA pairs"
        )
        
        print(f"  ✓ Uploaded {filename} ({len(standardized_data)} QA pairs)")
        
        # Cleanup
        os.remove(temp_filepath)
        
        # Log schema info
        print(f"  Schema: {list(df.columns)}")
        print(f"  Dtypes consistent: {df.dtypes.nunique() <= len(df.columns)}")
        
    except Exception as e:
        print(f"Failed to save standardized batch: {e}")
        import traceback
        traceback.print_exc()

def continue_corpus_synthesis():
    """Continue corpus synthesis with proper batch tracking"""
    corpus_ds = load_corpus_dataset()
    if not corpus_ds:
        return
    
    print(f"Corpus dataset: {len(corpus_ds)} total rows")
    current_row = progress_manager.progress_data.get('current_row', 0)
    
    if current_row >= len(corpus_ds):
        print("All corpus rows processed!")
        return
    
    print(f"Continuing from row {current_row}")
    process_corpus_batch(current_row)
    
    # Update progress
    progress_manager.progress_data['current_row'] = min(current_row + CONFIG['batch_size'], len(corpus_ds))
    progress_manager.save_progress()

def validate_dataset_files():
    """Validate that all uploaded files can be read by HF"""
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        
        files = api.list_repo_files(CONFIG['output_repository'], repo_type="dataset")
        parquet_files = [f for f in files if f.endswith('.parquet') and f.startswith('train-')]
        
        print(f"Dataset Validation:")
        print(f"  Found {len(parquet_files)} parquet files")
        print(f"  Files: {parquet_files}")
        
        # Try to load the dataset
        try:
            test_ds = load_dataset(CONFIG['output_repository'])
            print(f"  ✓ Dataset loads successfully")
            print(f"  Rows: {len(test_ds['train']) if 'train' in test_ds else len(test_ds)}")
            
            # Check schema consistency
            sample = test_ds['train'][0] if 'train' in test_ds else test_ds[0]
            print(f"  Schema: {list(sample.keys())}")
            
        except Exception as e:
            print(f"  ✗ Dataset loading failed: {e}")
            print("  This might be due to schema inconsistencies")
        
    except Exception as e:
        print(f"Validation failed: {e}")

In [ ]:
# Cell 11: Utility Functions
def show_corpus_config():
    """Display current corpus configuration"""
    print("Corpus Synthesis Configuration:")
    print(f"  Domain: {DOMAIN_CONFIG['domain']}")
    print(f"  Approach: {DOMAIN_CONFIG['approach']}")
    print(f"  Source Dataset: {CONFIG['source_dataset']}")
    print(f"  Corpus Column: {CONFIG['corpus_column']}")
    print(f"  Variants per text: {DOMAIN_CONFIG['num_variants']}")
    print(f"  Max text length: {DOMAIN_CONFIG['max_text_length']}")

def update_corpus_config(corpus_column=None, approach=None, num_variants=None, max_text_length=None):
    """Update corpus configuration"""
    if corpus_column:
        CONFIG['corpus_column'] = corpus_column
    if approach:
        DOMAIN_CONFIG['approach'] = approach
    if num_variants:
        DOMAIN_CONFIG['num_variants'] = num_variants
    if max_text_length:
        DOMAIN_CONFIG['max_text_length'] = max_text_length
    
    print("Updated corpus configuration")
    show_corpus_config()

def show_progress():
    """Show current progress with enhanced file validation"""
    stats = progress_manager.progress_data
    print(f"Corpus Synthesis Progress:")
    print(f"  Current row: {stats.get('current_row', 0)}")
    print(f"  Processed rows: {len(stats.get('processed_rows', []))}")
    print(f"  QA pairs created: {stats['total_qa_pairs_created']}")
    print(f"  API requests: {stats['request_count']}")
    
    # Validate HF dataset compatibility
    validate_dataset_files()

# Keep the rest of Cell 11 as is...

In [ ]:
# Initialize progress manager
progress_manager = ProgressManager()

print("Corpus-Based QA Synthesis Pipeline Ready!")
print("=" * 50)
print("Key Functions:")
print("- show_corpus_config() - Display current configuration")
print("- update_corpus_config(corpus_column, chunk_method, chunk_size, approach) - Update settings")
print("- continue_corpus_synthesis() - Start/continue synthesis")
print("- process_corpus_batch(start_row, end_row) - Process specific rows")
print("- show_progress() - Check current progress")
print("\nFirst, update CONFIG with your dataset info, then run: continue_corpus_synthesis()")

show_corpus_config()
continue_corpus_synthesis()